In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL ='groq/llama-3.3-70b-versatile'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:56<00:00, 18.77s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

"Title: SanDisk Memory Deals at Woot for From $10 + free shipping w/ Prime\nDetails: These are all factory reconditioned and come with a 90-day Woot limited warranty. We've pictured the SanDisk Ultra Plus 512GB microSDXC Memory Card for $35 ($45 less than you'd pay for a factory sealed version). Buy Now at Woot! An Amazon Company\nFeatures: \nURL: https://www.dealnews.com/San-Disk-Memory-Deals-at-Woot-for-From-10-free-shipping-w-Prime/21811819.html?iref=rss-c39"

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Refurb Samsung Galaxy Watch5 Pro 45mm GPS Smartwatch for $57 + free shipping
Details: It ties the best deal we've seen for this model in any condition. It includes a 1-year Allstate warranty.  Buy Now at eBay
Features: water resistant to 164' 45mm AMOLED display measures blood oxygen, electrocardiography, glucose, heart rate, hours slept Android Wear OS Model: SM-R920NZTAXAA
UR

In [8]:
import json
import time
from litellm import completion
from pydantic import ValidationError


def groq_parse(messages, schema, retries=3):
    format_instruction = {
        "role": "system",
        "content": """
Return ONLY valid JSON in this exact format:

{
  "deals": [
    {
      "product_description": "string",
      "price": number,
      "url": "string"
    }
  ]
}

Include exactly 5 deals.
No markdown.
No explanation.
Raw JSON only.
"""
    }

    for attempt in range(retries):
        response = completion(
            model="groq/llama-3.3-70b-versatile",
            messages=[format_instruction] + messages,
            temperature=0,
        )

        content = response.choices[0].message.content.strip()

        try:
            data = json.loads(content)
            return schema(**data)
        except (json.JSONDecodeError, ValidationError):
            print(f"Retry {attempt+1}: Invalid JSON, retrying...")
            time.sleep(1)

    raise ValueError("Model failed to produce valid structured output.")

In [9]:

result = groq_parse(messages, DealSelection)

print(result.deals)
# If you have billing gpt account use this but dont forget to change the model to 'gpt-5-mini'
# response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
# results = response.choices[0].message.parsed
# results

[Deal(product_description='The Samsung Galaxy Watch5 Pro 45mm GPS Smartwatch is a water-resistant smartwatch that features a 45mm AMOLED display, measures blood oxygen, electrocardiography, glucose, heart rate, and hours slept. It runs on Android Wear OS and has a range of features to track your health and fitness.', price=57.0, url='https://www.dealnews.com/products/Samsung/Samsung-Galaxy-Watch5-Pro-45-mm-GPS-Smartwatch/386240.html?iref=rss-c142'), Deal(product_description='The Apple Watch Series 8 GPS + GSM Cellular 41mm Smart Watch is a sleek and feature-packed smartwatch that includes a range of health and fitness tracking features, as well as the ability to make and receive calls and texts. It also includes a one-year Allstate warranty.', price=175.0, url='https://www.dealnews.com/products/Apple/Apple-Watch-Series-8-GPS-GSM-Cellular-41-mm-Smart-Watch/468790.html?iref=rss-c142'), Deal(product_description='The Bose Solo Soundbar II is a compact soundbar that features Bluetooth conne

In [10]:
for deal in result.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The Samsung Galaxy Watch5 Pro 45mm GPS Smartwatch is a water-resistant smartwatch that features a 45mm AMOLED display, measures blood oxygen, electrocardiography, glucose, heart rate, and hours slept. It runs on Android Wear OS and has a range of features to track your health and fitness.
57.0
https://www.dealnews.com/products/Samsung/Samsung-Galaxy-Watch5-Pro-45-mm-GPS-Smartwatch/386240.html?iref=rss-c142

The Apple Watch Series 8 GPS + GSM Cellular 41mm Smart Watch is a sleek and feature-packed smartwatch that includes a range of health and fitness tracking features, as well as the ability to make and receive calls and texts. It also includes a one-year Allstate warranty.
175.0
https://www.dealnews.com/products/Apple/Apple-Watch-Series-8-GPS-GSM-Cellular-41-mm-Smart-Watch/468790.html?iref=rss-c142

The Bose Solo Soundbar II is a compact soundbar that features Bluetooth connectivity, allowing you to stream music from your phone or tablet. It also includes a remote control and has a ra

In [11]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [12]:
from agents.scanner_agent import ScannerAgent

In [13]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing (Groq mode)
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling Groq
22:39:12 - LiteLLM:INFO: utils.py:3879 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
22:39:14 - LiteLLM:INFO: utils.py:1629 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from Groq


In [14]:
result

DealSelection(deals=[Deal(product_description="Samsung Galaxy Watch5 Pro 45mm GPS Smartwatch with water resistance to 164', 45mm AMOLED display, and features to measure blood oxygen, electrocardiography, glucose, heart rate, and hours slept", price=57.0, url='https://www.dealnews.com/products/Samsung/Samsung-Galaxy-Watch5-Pro-45-mm-GPS-Smartwatch/386240.html?iref=rss-c142'), Deal(product_description='Certified Refurb Bose Solo Soundbar II with Bluetooth range up to 30-feet, remote control, and measures 21.6 x 2.8', price=99.0, url='https://www.dealnews.com/products/Bose/Bose-Solo-Soundbar-II/389279.html?iref=rss-c142'), Deal(product_description='Yogasleep Dohm Classic White Noise Machine with 7-foot 120V AC power cable and fan-based white noise', price=35.0, url='https://www.dealnews.com/Yogasleep-Dohm-Classic-White-Noise-Machine-for-35-free-shipping/21811821.html?iref=rss-c142'), Deal(product_description='Refurb Samsung QN50LS03DA The Frame 50 4K QLED Smart TV with 4K resolution, HDR 

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [2]:
load_dotenv(override=True)

True

In [3]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [12]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [13]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [14]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

: 